In [3]:

# ================================================================
# JEEVAN LINK - ASSISTANCE PREDICTION MODEL
# COMPLETE TRAINING AND EVALUATION PIPELINE
# ================================================================

# 1. IMPORT LIBRARIES
# ================================================================

import pandas as pd
import numpy as np
import json
import joblib
import warnings

from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    classification_report
)

warnings.filterwarnings("ignore")

print("=" * 70)
print("JEEVAN LINK ASSISTANCE MODEL TRAINING")
print("=" * 70)


# 2. PROJECT PATHS
# ================================================================

PROJECT_DIR = Path(
    r"C:\Users\ishit\jeevan_link_ml"
)

DATA_PATH = (
    PROJECT_DIR / "Data" / "Processed_data" /
    "Assitance_Dataset_for_model" /
    "FINAL_ASSISTANCE_TRAINING_DATASET" /
    "assistance_dataset_training_ready.csv"
)

MODEL_DIR = (
    PROJECT_DIR / "Models" / "Assistance_Model"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Check the CSV filename and folder."
    )


# 3. LOAD DATASET
# ================================================================

df = pd.read_csv(DATA_PATH)

LABELS = [
    "food",
    "water",
    "shelter",
    "medical_help",
    "medical_products",
    "search_and_rescue",
    "transport"
]

required_columns = [
    "message", "source_split", *LABELS
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )

df["message"] = df["message"].fillna("").astype(str)

df["source_split"] = (
    df["source_split"].astype(str).str.strip().str.lower()
)

if df["message"].str.strip().eq("").any():
    raise ValueError("Empty messages found in dataset.")

for label in LABELS:

    df[label] = pd.to_numeric(
        df[label], errors="raise"
    )

    if df[label].isna().any():
        raise ValueError(f"Missing labels found: {label}")

    if not df[label].isin([0, 1]).all():
        raise ValueError(f"Invalid binary labels: {label}")

    df[label] = df[label].astype(int)

print("\nDATASET LOADED")
print("Dataset shape:", df.shape)

print("\nSplit distribution:")
print(df["source_split"].value_counts())


# 4. CREATE TRAIN, VALIDATION AND TEST SPLITS
# ================================================================

train_mask = df["source_split"] == "train"
val_mask = df["source_split"] == "validation"
test_mask = df["source_split"] == "test"

if not train_mask.any() or not val_mask.any() or not test_mask.any():
    raise ValueError("Train, validation or test split is empty.")

X_train = df.loc[train_mask, "message"]
Y_train = df.loc[train_mask, LABELS]

X_val = df.loc[val_mask, "message"]
Y_val = df.loc[val_mask, LABELS]

X_test = df.loc[test_mask, "message"]
Y_test = df.loc[test_mask, LABELS]

print("\nSPLIT SIZES")
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

if (Y_train.sum(axis=0) == 0).any():
    raise ValueError(
        "At least one label has no positive training examples."
    )


# 5. TF-IDF TEXT VECTORIZATION
# ================================================================

print("\nConverting messages into TF-IDF features...")

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    strip_accents="unicode",
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF completed.")
print("Training feature matrix:", X_train_tfidf.shape)
print("Validation feature matrix:", X_val_tfidf.shape)
print("Test feature matrix:", X_test_tfidf.shape)



# ================================================================
# STEP 6 - FIXED ASSISTANCE MODEL TRAINING
# ================================================================

from sklearn.base import clone
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

print("Preparing writable TF-IDF training matrix...")

# Create independent writable copies
X_train_tfidf = X_train_tfidf.copy()
X_train_tfidf.sort_indices()

Y_train = Y_train.copy()

# Base classifier
base_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="liblinear",
    random_state=42
)

# Disable parallel processing to avoid Windows
# multiprocessing and sparse-matrix write conflicts
assistance_model = OneVsRestClassifier(
    base_model,
    n_jobs=1
)

print("Training Assistance model...")

assistance_model.fit(
    X_train_tfidf,
    Y_train
)

print("\nASSISTANCE MODEL TRAINED SUCCESSFULLY!")
print("Number of trained classifiers:", len(assistance_model.estimators_))
print("Number of assistance categories:", len(LABELS))



# 7. VALIDATION-BASED THRESHOLD SELECTION
# ================================================================

print("\nSelecting thresholds using validation data...")

val_probabilities = assistance_model.predict_proba(
    X_val_tfidf
)

thresholds = {}

for i, label in enumerate(LABELS):

    best_threshold = 0.50
    best_f1 = -1

    for threshold in np.arange(0.10, 0.91, 0.01):

        predictions = (
            val_probabilities[:, i] >= threshold
        ).astype(int)

        score = f1_score(
            Y_val.iloc[:, i],
            predictions,
            zero_division=0
        )

        if score > best_f1:
            best_f1 = score
            best_threshold = threshold

    thresholds[label] = round(float(best_threshold), 2)

print("\nSELECTED THRESHOLDS")

for label, threshold in thresholds.items():
    print(f"{label}: {threshold}")


# 8. VALIDATION EVALUATION
# ================================================================

val_predictions = np.column_stack([
    (
        val_probabilities[:, i] >= thresholds[label]
    ).astype(int)
    for i, label in enumerate(LABELS)
])

validation_metrics = {
    "subset_accuracy": accuracy_score(
        Y_val, val_predictions
    ),
    "micro_precision": precision_score(
        Y_val, val_predictions,
        average="micro", zero_division=0
    ),
    "micro_recall": recall_score(
        Y_val, val_predictions,
        average="micro", zero_division=0
    ),
    "micro_f1": f1_score(
        Y_val, val_predictions,
        average="micro", zero_division=0
    ),
    "macro_f1": f1_score(
        Y_val, val_predictions,
        average="macro", zero_division=0
    ),
    "hamming_loss": hamming_loss(
        Y_val, val_predictions
    )
}

print("\n" + "=" * 70)
print("VALIDATION PERFORMANCE")
print("=" * 70)

for metric, value in validation_metrics.items():
    print(f"{metric}: {value:.4f}")


# 9. FINAL TEST EVALUATION
# ================================================================

print("\nEvaluating on untouched test data...")

test_probabilities = assistance_model.predict_proba(
    X_test_tfidf
)

test_predictions = np.column_stack([
    (
        test_probabilities[:, i] >= thresholds[label]
    ).astype(int)
    for i, label in enumerate(LABELS)
])

test_metrics = {
    "subset_accuracy": accuracy_score(
        Y_test, test_predictions
    ),
    "micro_precision": precision_score(
        Y_test, test_predictions,
        average="micro", zero_division=0
    ),
    "micro_recall": recall_score(
        Y_test, test_predictions,
        average="micro", zero_division=0
    ),
    "micro_f1": f1_score(
        Y_test, test_predictions,
        average="micro", zero_division=0
    ),
    "macro_precision": precision_score(
        Y_test, test_predictions,
        average="macro", zero_division=0
    ),
    "macro_recall": recall_score(
        Y_test, test_predictions,
        average="macro", zero_division=0
    ),
    "macro_f1": f1_score(
        Y_test, test_predictions,
        average="macro", zero_division=0
    ),
    "weighted_f1": f1_score(
        Y_test, test_predictions,
        average="weighted", zero_division=0
    ),
    "hamming_loss": hamming_loss(
        Y_test, test_predictions
    )
}

print("\n" + "=" * 70)
print("FINAL TEST PERFORMANCE")
print("=" * 70)

for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")


# 10. PER-LABEL CLASSIFICATION REPORT
# ================================================================

print("\nPER-LABEL CLASSIFICATION REPORT")

print(
    classification_report(
        Y_test,
        test_predictions,
        target_names=LABELS,
        zero_division=0
    )
)

report = classification_report(
    Y_test,
    test_predictions,
    target_names=LABELS,
    output_dict=True,
    zero_division=0
)


# 11. SAVE TRAINED MODEL
# ================================================================

print("\nSaving trained model files...")

joblib.dump(
    assistance_model,
    MODEL_DIR / "assistance_model.pkl"
)

joblib.dump(
    tfidf,
    MODEL_DIR / "assistance_tfidf_vectorizer.pkl"
)

joblib.dump(
    LABELS,
    MODEL_DIR / "assistance_label_names.pkl"
)

with open(
    MODEL_DIR / "assistance_thresholds.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(thresholds, f, indent=4)


# 12. SAVE EVALUATION METRICS
# ================================================================

with open(
    MODEL_DIR / "assistance_model_metrics.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "validation_metrics": {
                k: float(v)
                for k, v in validation_metrics.items()
            },
            "test_metrics": {
                k: float(v)
                for k, v in test_metrics.items()
            }
        },
        f,
        indent=4
    )

pd.DataFrame(report).transpose().to_csv(
    MODEL_DIR / "assistance_classification_report.csv"
)

pd.DataFrame(
    list(thresholds.items()),
    columns=["label", "threshold"]
).to_csv(
    MODEL_DIR / "assistance_thresholds.csv",
    index=False
)


# 13. SAVE TEST PREDICTIONS
# ================================================================

test_results = df.loc[
    test_mask, ["message"]
].copy()

for i, label in enumerate(LABELS):

    test_results[label + "_actual"] = (
        Y_test.iloc[:, i].values
    )

    test_results[label + "_predicted"] = (
        test_predictions[:, i]
    )

    test_results[label + "_probability"] = (
        test_probabilities[:, i]
    )

test_results.to_csv(
    MODEL_DIR / "assistance_test_predictions.csv",
    index=False
)


# 14. CREATE A REUSABLE PREDICTION FUNCTION
# ================================================================

def predict_assistance(message):

    message = str(message).strip()

    if not message:
        raise ValueError("Please provide a non-empty message.")

    features = tfidf.transform([message])

    probabilities = assistance_model.predict_proba(
        features
    )[0]

    results = []

    for i, label in enumerate(LABELS):

        probability = float(probabilities[i])

        predicted = (
            probability >= thresholds[label]
        )

        results.append({
            "assistance_category": label,
            "probability": round(probability, 4),
            "predicted": bool(predicted)
        })

    return pd.DataFrame(results)


# Save prediction function dependencies as model artifacts.
# The function itself can be recreated in the application
# using the saved model, vectorizer, labels and thresholds.


# 15. TEST WITH SAMPLE EMERGENCY MESSAGE
# ================================================================

sample_message = (
    "Our family needs clean drinking water, food "
    "and medical help after the flood."
)

print("\nSAMPLE EMERGENCY PREDICTION")

display(
    predict_assistance(sample_message)
)


# 16. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 70)
print("ASSISTANCE MODEL TRAINING COMPLETED")
print("=" * 70)

print("Dataset records:", len(df))
print("Training records:", len(X_train))
print("Validation records:", len(X_val))
print("Testing records:", len(X_test))
print("Number of assistance categories:", len(LABELS))

print("\nSaved files:")

for file in sorted(MODEL_DIR.iterdir()):
    if file.is_file():
        print(" -", file.name)

print("\nMODEL OUTPUT DIRECTORY:")
print(MODEL_DIR)

print("\nTraining and evaluation pipeline finished.")


JEEVAN LINK ASSISTANCE MODEL TRAINING

DATASET LOADED
Dataset shape: (10658, 11)

Split distribution:
source_split
train         8571
test          1055
validation    1032
Name: count, dtype: int64

SPLIT SIZES
Training: 8571
Validation: 1032
Testing: 1055

Converting messages into TF-IDF features...
TF-IDF completed.
Training feature matrix: (8571, 16696)
Validation feature matrix: (1032, 16696)
Test feature matrix: (1055, 16696)
Preparing writable TF-IDF training matrix...
Training Assistance model...

ASSISTANCE MODEL TRAINED SUCCESSFULLY!
Number of trained classifiers: 7
Number of assistance categories: 7

Selecting thresholds using validation data...

SELECTED THRESHOLDS
food: 0.58
water: 0.59
shelter: 0.58
medical_help: 0.65
medical_products: 0.63
search_and_rescue: 0.51
transport: 0.29

VALIDATION PERFORMANCE
subset_accuracy: 0.6793
micro_precision: 0.5110
micro_recall: 0.6764
micro_f1: 0.5822
macro_f1: 0.4865
hamming_loss: 0.0694

Evaluating on untouched test data...

FINAL TES

,assistance_category,probability,predicted
0,food,0.9583,True
1,water,0.9692,True
2,shelter,0.6902,True
3,medical_help,0.8719,True
4,medical_products,0.8621,True
5,search_and_rescue,0.7599,True
6,transport,0.3891,True



ASSISTANCE MODEL TRAINING COMPLETED
Dataset records: 10658
Training records: 8571
Validation records: 1032
Testing records: 1055
Number of assistance categories: 7

Saved files:
 - assistance_classification_report.csv
 - assistance_label_names.pkl
 - assistance_model.pkl
 - assistance_model_metrics.json
 - assistance_test_predictions.csv
 - assistance_tfidf_vectorizer.pkl
 - assistance_thresholds.csv
 - assistance_thresholds.json

MODEL OUTPUT DIRECTORY:
C:\Users\ishit\jeevan_link_ml\Models\Assistance_Model

Training and evaluation pipeline finished.
